In [2]:
import os
os.chdir("../")
%pwd

'c:\\Users\\xuanl\\ChickenDisease-hx'

In [3]:
# Update Config.yaml
# update Entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig: 
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [4]:
# Update Configurations manager
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [5]:
class ConfigurationManager: 
    def __init__(
            self, 
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH): 
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callback_config
    
    def get_training_config(self) -> TrainingConfig: 
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chicken-fecal-images")
        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(
            root_dir = Path(training.root_dir), 
            trained_model_path = Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data = Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
        )

        return training_config

In [6]:
# Update Components
import time

In [7]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config


    
    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True
        )


    def get_tb_ckpt_callbacks(self):
        return [
            self._create_tb_callbacks,
            self._create_ckpt_callbacks
        ]

In [8]:
# Create new component

import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
class Training: 
    def __init__(self, config: TrainingConfig): 
        self.config = config
    
    def get_base_model(self): 
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self): 
        
        datagenerator_kawrgs = dict(
            rescale = 1./255,
            validation_split = 0.20
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = "bilinear"
        )

        valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kawrgs
        )

        self.valid_generator = valid_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False, 
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation: 
            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2, 
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kawrgs
            )
        else: 
            train_datagen = valid_datagen

        self.train_generator = train_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )
    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model): 
        model.save(path)

    def train(self, callback_list: list): 
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.history = self.model.fit(
            self.train_generator, 
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps = self.validation_steps,
            callbacks = callback_list
        )

        self.save_model(
            path=self.config.trained_model_path,
            model = self.model
        )

In [10]:
# Create pipeline
try: 
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train(callback_list = callback_list)

except Exception as e:
    raise e

[2026-06-18 03:09:28,575: INFO: common] yaml file: config\config.yaml loaded successfully
[2026-06-18 03:09:28,579: INFO: common] yaml file: params.yaml loaded successfully
[2026-06-18 03:09:28,581: INFO: common] directory created at: artifacts
[2026-06-18 03:09:28,582: INFO: common] directory created at: artifacts\prepare_callbacks\checkpoint_dir
[2026-06-18 03:09:28,584: INFO: common] directory created at: artifacts\prepare_callbacks\tensorboard_log_dir
[2026-06-18 03:09:28,587: INFO: common] directory created at: artifacts\training
[2026-06-18 03:09:28,887: WARNING: saving_utils] Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.


c:\Users\xuanl\Downloads\Anaconda\envs\MUAI-class\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


NotImplementedError: numpy() is only available when eager execution is enabled.

In [11]:
import tensorflow as tf
import numpy as np
import math
from pathlib import Path
from cnnClassifier.entity.config_entity import TrainingConfig


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    # --------------------------
    # Load base model
    # --------------------------
    def get_base_model(self):
        print("Loading model from:", self.config.updated_base_model_path)

        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=True
        )

    # --------------------------
    # Data generators
    # --------------------------
    def train_valid_generator(self):

        datagen_args = dict(
            rescale=1. / 255,
            validation_split=0.20
        )

        dataflow_args = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        # Validation generator (NO augmentation)
        valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagen_args
        )

        self.valid_generator = valid_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_args
        )

        # Training generator (WITH augmentation if enabled)
        if self.config.params_is_augmentation:
            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagen_args
            )
        else:
            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                **datagen_args
            )

        self.train_generator = train_datagen.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_args
        )

    # --------------------------
    # Save model
    # --------------------------
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    # --------------------------
    # Train model
    # --------------------------
    def train(self, callback_list: list):

        # SAFE steps calculation (avoid 0 crash)
        self.steps_per_epoch = math.ceil(
            self.train_generator.samples / self.train_generator.batch_size
        )

        self.validation_steps = math.ceil(
            self.valid_generator.samples / self.valid_generator.batch_size
        )

        print("Steps per epoch:", self.steps_per_epoch)
        print("Validation steps:", self.validation_steps)

        # Train
        self.history = self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            callbacks=callback_list
        )

        # Save model
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [12]:
try:
    config = ConfigurationManager()

    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()

    training_config = config.get_training_config()
    training = Training(config=training_config)

    training.get_base_model()
    training.train_valid_generator()
    training.train(callback_list=callback_list)

except Exception as e:
    raise e

[2026-06-18 03:12:32,469: INFO: common] yaml file: config\config.yaml loaded successfully
[2026-06-18 03:12:32,472: INFO: common] yaml file: params.yaml loaded successfully
[2026-06-18 03:12:32,475: INFO: common] directory created at: artifacts
[2026-06-18 03:12:32,478: INFO: common] directory created at: artifacts\prepare_callbacks\checkpoint_dir
[2026-06-18 03:12:32,480: INFO: common] directory created at: artifacts\prepare_callbacks\tensorboard_log_dir
[2026-06-18 03:12:32,481: INFO: common] directory created at: artifacts\training
Loading model from: artifacts\prepare_base_model\base_model_udpated.h5
[2026-06-18 03:12:32,665: WARNING: saving_utils] Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
Found 78 images belonging to 2 classes.
Found 312 images belonging to 2 classes.
Steps per epoch: 20
Validation steps: 5
Epoch 1/10


NotImplementedError: numpy() is only available when eager execution is enabled.

In [13]:
tf.compat.v1.disable_eager_execution()

[2026-06-18 03:13:08,831: WARNING: module_wrapper] From C:\Users\xuanl\AppData\Local\Temp\ipykernel_24216\45053954.py:1: The name tf.disable_eager_execution is deprecated. Please use tf.compat.v1.disable_eager_execution instead.



In [14]:
import tensorflow as tf

print("TF Version:", tf.__version__)
print("Eager execution:", tf.executing_eagerly())

TF Version: 2.20.0
Eager execution: False
